# EduVision_DV — 03. Standardization & Master IDs

In [3]:
import pandas as pd
import os
from rapidfuzz import process, fuzz

CLEANED_DIR = os.path.join(".", "..", "data", "cleaned")
DOCS_DIR = os.path.join(".", "..", "docs")

KNOWN_FALSE_POSITIVES = {
    "northeastern university", "southwest university", "national university",
    "national university of ireland", "university of southern queensland",
}

SIM_STRONG = 95
SIM_REVIEW_LOW = 90


def load_sources():
    qs = pd.read_csv(os.path.join(CLEANED_DIR, "qs_2025_clean.csv"))
    the = pd.read_csv(os.path.join(CLEANED_DIR, "the_2024_clean.csv"))
    wur = pd.read_csv(os.path.join(CLEANED_DIR, "wur_2023_clean.csv"))

    qs = qs.rename(columns={"institution_name": "university_name_raw"})
    the = the.rename(columns={"name": "university_name_raw"})
    wur = wur.rename(columns={"name_of_university": "university_name_raw"})

    qs["source"] = "QS_2025"
    the["source"] = "THE_2024"
    wur["source"] = "WUR_2023"
    return qs, the, wur


def match_two_sources(base, other, base_label, other_label):
    """Exact match on (standardized name, country) first, then a
    country-scoped fuzzy match for the leftovers - candidates only."""
    base = base.copy()
    other = other.copy()
    base["key"] = base["university_name_clean"] + "|" + base["country_clean"].astype(str)
    other["key"] = other["university_name_clean"] + "|" + other["country_clean"].astype(str)

    common = set(base["key"]) & set(other["key"])
    other_lookup = other.drop_duplicates("key").set_index("key")["university_name_raw"].to_dict()

    exact_rows = base[base["key"].isin(common)]
    direct = pd.DataFrame({
        f"{base_label}_name": exact_rows["university_name_raw"],
        "country": exact_rows["country_clean"],
        f"{other_label}_name": exact_rows["key"].map(other_lookup),
        "similarity": 100.0,
        "match_type": "Exact",
    })

    unmatched = base[~base["key"].isin(common)]
    other_by_country = {
        c: g[["university_name_clean", "university_name_raw"]].values.tolist()
        for c, g in other.groupby("country_clean")
    }
    fuzzy_rows = []
    for _, row in unmatched.iterrows():
        cands = other_by_country.get(row["country_clean"], [])
        if not cands:
            continue
        cand_std = [c[0] for c in cands]
        m = process.extractOne(row["university_name_clean"], cand_std, scorer=fuzz.token_sort_ratio)
        if m:
            std_name, score, idx = m
            fuzzy_rows.append({
                f"{base_label}_name": row["university_name_raw"],
                "country": row["country_clean"],
                f"{other_label}_name": cands[idx][1],
                "similarity": score,
                "match_type": "Fuzzy",
            })
    fuzzy = pd.DataFrame(fuzzy_rows)
    if len(fuzzy):
        fuzzy["quality"] = fuzzy["similarity"].apply(
            lambda s: "Strong" if s >= SIM_STRONG else ("Review" if s >= SIM_REVIEW_LOW else "Weak")
        )
        strong = fuzzy[fuzzy["quality"] == "Strong"].copy()
        strong = strong[~strong[f"{base_label}_name"].str.lower().str.strip().isin(KNOWN_FALSE_POSITIVES)]
        review = fuzzy[fuzzy["quality"] == "Review"]
    else:
        strong, review = fuzzy, fuzzy

    accepted = pd.concat([
        direct,
        strong.drop(columns=["quality"], errors="ignore") if len(strong) else strong,
    ], ignore_index=True) if len(strong) or len(direct) else direct

    return accepted, review


def build_dim_university(qs, the, wur):
    print("=" * 60, "\nCROSS-SOURCE MATCHING (candidates only - see doc for manual checks)\n" + "=" * 60)

    qt_matches, qt_review = match_two_sources(qs, the, "qs", "the")
    print(f"QS<->THE: {len(qt_matches)} accepted matches, {len(qt_review)} flagged for manual review")

    qw_matches, qw_review = match_two_sources(qs, wur, "qs", "wur")
    print(f"QS<->WUR2023: {len(qw_matches)} accepted matches, {len(qw_review)} flagged for manual review")

    the_map = qt_matches.drop_duplicates("qs_name").set_index("qs_name")["the_name"].to_dict()
    wur_map = qw_matches.drop_duplicates("qs_name").set_index("qs_name")["wur_name"].to_dict()

    qs_master = qs[["university_name_raw", "university_name_clean", "country_clean", "region"]].copy()
    qs_master = qs_master.rename(columns={"university_name_raw": "qs_name"})
    qs_master["the_name"] = qs_master["qs_name"].map(the_map)
    qs_master["wur_name"] = qs_master["qs_name"].map(wur_map)
    qs_master["in_qs"] = True
    qs_master["in_the"] = qs_master["the_name"].notna()
    qs_master["in_wur"] = qs_master["wur_name"].notna()

    matched_the_names = set(qt_matches["the_name"].dropna())
    the_only = the[~the["university_name_raw"].isin(matched_the_names)][
        ["university_name_raw", "university_name_clean", "country_clean"]
    ].copy()
    the_only = the_only.rename(columns={"university_name_raw": "the_name"})
    the_only["qs_name"] = pd.NA
    the_only["wur_name"] = pd.NA
    the_only["region"] = pd.NA
    the_only["in_qs"], the_only["in_the"], the_only["in_wur"] = False, True, False

    matched_wur_names = set(qw_matches["wur_name"].dropna())
    wur_only = wur[~wur["university_name_raw"].isin(matched_wur_names)][
        ["university_name_raw", "university_name_clean", "country_clean"]
    ].copy()
    wur_only = wur_only.rename(columns={"university_name_raw": "wur_name"})
    wur_only["qs_name"] = pd.NA
    wur_only["the_name"] = pd.NA
    wur_only["region"] = pd.NA
    wur_only["in_qs"], wur_only["in_the"], wur_only["in_wur"] = False, False, True

    dim_university = pd.concat([qs_master, the_only, wur_only], ignore_index=True)
    dim_university["university_id"] = ["U" + str(i + 1).zfill(5) for i in range(len(dim_university))]
    dim_university["display_name"] = dim_university["qs_name"].fillna(
        dim_university["the_name"]).fillna(dim_university["wur_name"])

    cols = ["university_id", "display_name", "country_clean", "region",
            "qs_name", "the_name", "wur_name", "in_qs", "in_the", "in_wur"]
    dim_university = dim_university[cols].rename(columns={"country_clean": "country_name"})

    print(f"\ndim_university rows: {len(dim_university)}")
    print(dim_university[["in_qs", "in_the", "in_wur"]].value_counts().to_string())

    return dim_university, qt_review, qw_review


def build_dim_country(dim_university):
    countries = sorted(dim_university["country_name"].dropna().unique())
    dim_country = pd.DataFrame({
        "country_id": [c[:2].upper() + str(i).zfill(2) if False else f"C{str(i+1).zfill(3)}"
                        for i, c in enumerate(countries)],
        "country_name": countries,
    })
    return dim_country


# ----------------------------------------------------------------------
# MAJOR POINT 5 — Match World Bank countries onto the same country_id
# ----------------------------------------------------------------------
def match_world_bank_countries(dim_country):
    print("\n" + "=" * 60, "\nMAJOR POINT 5: MATCH WORLD BANK COUNTRIES TO country_id\n" + "=" * 60)
    wb_path = os.path.join(CLEANED_DIR, "world_bank_education_clean.csv")
    if not os.path.exists(wb_path):
        print("world_bank_education_clean.csv not found yet (Step 2 skipped it) - "
              "nothing to match. Every other part of the model is unaffected.")
        return None

    wb = pd.read_csv(wb_path)
    dim_country_keyed = dim_country.copy()
    dim_country_keyed["country_clean"] = dim_country_keyed["country_name"].str.lower().str.strip()

    merged = wb.merge(dim_country_keyed[["country_id", "country_clean"]], on="country_clean", how="left")
    matched = merged["country_id"].notna().sum()
    total = merged["country_clean"].nunique()
    print(f"World Bank country-rows matched to an existing country_id: "
          f"{matched} / {len(merged)} ({merged.loc[merged['country_id'].notna(), 'country_clean'].nunique()} / {total} distinct countries)")

    unmatched = sorted(merged.loc[merged["country_id"].isna(), "country_clean"].unique())
    if unmatched:
        print(f"\nUnmatched World Bank country names ({len(unmatched)}) - logged for manual review, "
              f"NOT force-matched (per Section 7's rule: a false match is worse than a missing one):")
        for name in unmatched[:20]:
            print("  -", name)

    out = os.path.join(CLEANED_DIR, "world_bank_matched.csv")
    merged.to_csv(out, index=False)
    print(f"\nsaved -> {out}")
    return merged


if __name__ == "__main__":
    qs, the, wur = load_sources()
    dim_university, qt_review, qw_review = build_dim_university(qs, the, wur)
    dim_country = build_dim_country(dim_university)

    dim_university = dim_university.merge(
        dim_country, on="country_name", how="left"
    ).drop(columns=["country_name"])
    # re-attach country_name for readability alongside the id (both are useful in Tableau)
    dim_university = dim_university.merge(dim_country, on="country_id", how="left")

    dim_university.to_csv(os.path.join(CLEANED_DIR, "dim_university.csv"), index=False)
    dim_country.to_csv(os.path.join(CLEANED_DIR, "dim_country.csv"), index=False)
    print(f"\nsaved -> dim_university.csv ({len(dim_university)} rows)")
    print(f"saved -> dim_country.csv ({len(dim_country)} rows)")

    review_all = pd.concat([
        qt_review.assign(pair="QS<->THE") if len(qt_review) else qt_review,
        qw_review.assign(pair="QS<->WUR2023") if len(qw_review) else qw_review,
    ], ignore_index=True)
    review_path = os.path.join(CLEANED_DIR, "matches_for_manual_review.csv")
    review_all.to_csv(review_path, index=False)
    print(f"saved -> matches_for_manual_review.csv ({len(review_all)} rows, 90-94.9% similarity, NOT auto-merged)")

    match_world_bank_countries(dim_country)

    os.makedirs(DOCS_DIR, exist_ok=True)
    with open(os.path.join(DOCS_DIR, "02_matching_methodology.md"), "w") as f:
        f.write(f"""# Matching Methodology

## Rule
Fuzzy matching identifies *candidates* only. A match is only written into
`dim_university` if:
1. Exact match on (standardized name, standardized country), OR
2. Fuzzy similarity >= {SIM_STRONG} (rapidfuzz token_sort_ratio) AND same country,
   AND not in the manually-reviewed false-positive list.

Matches scoring {SIM_REVIEW_LOW}-{SIM_STRONG - 0.1} similarity are written to
`matches_for_manual_review.csv` and are **not** merged automatically - a
false match is worse than a missing match (per project reference guide,
Section 7).

## Known false positives (excluded even at >=95% similarity)
{chr(10).join('- ' + n for n in sorted(KNOWN_FALSE_POSITIVES))}

## Results
- dim_university rows: {len(dim_university)}
- QS universities: {dim_university['in_qs'].sum()}
- THE universities: {dim_university['in_the'].sum()}
- WUR 2023 universities: {dim_university['in_wur'].sum()}
- Present in all three sources: {((dim_university['in_qs']) & (dim_university['in_the']) & (dim_university['in_wur'])).sum()}
""")
    print("saved -> docs/02_matching_methodology.md")


CROSS-SOURCE MATCHING (candidates only - see doc for manual checks)
QS<->THE: 934 accepted matches, 82 flagged for manual review
QS<->WUR2023: 767 accepted matches, 64 flagged for manual review

dim_university rows: 4708
in_qs  in_the  in_wur
False  True    False     1739
       False   True      1466
True   True    True       742
       False   False      544
       True    False      192
       False   True        25

saved -> dim_university.csv (4708 rows)
saved -> dim_country.csv (136 rows)
saved -> matches_for_manual_review.csv (146 rows, 90-94.9% similarity, NOT auto-merged)

MAJOR POINT 5: MATCH WORLD BANK COUNTRIES TO country_id
World Bank country-rows matched to an existing country_id: 557 / 1004 (119 / 232 distinct countries)

Unmatched World Bank country names (113) - logged for manual review, NOT force-matched (per Section 7's rule: a false match is worse than a missing one):
  - afghanistan
  - american samoa
  - andorra
  - antigua and barbuda
  - arab world
  - aruba
  -